# 07 - Reader: Text and Image on Dev Set

This notebook tests our reader model on the 30 dev questions in 5 different settings:

| Setting | What the model gets |
|---|---|
| `gold_text` | The text of the evidence pages |
| `gold_image` | Images of the evidence pages |
| `retrieved_text` | The text of the top 5 pages of fused + rerank |
| `retrieved_image` | Images of the top 5 pages of fused + rerank |
| `closed_book` | Only the question |

The aim is to find problems such as memory limits or cut-off answers before running on 120 test questions.

This notebook runs on Google Colab.

In [1]:
!git clone https://github.com/dogamentese/financebench-rag.git
%cd financebench-rag
!pip install -q -U transformers accelerate
!pip install -q pymupdf
!python scripts/download_data.py

Cloning into 'financebench-rag'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 71 (delta 29), reused 57 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 739.90 KiB | 14.23 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/financebench-rag
[1/84] 3M_2018_10K
[2/84] 3M_2022_10K
[3/84] 3M_2023Q2_10Q
[4/84] ACTIVISIONBLIZZARD_2019_10K
[5/84] ADOBE_2015_10K
[6/84] ADOBE_2016_10K
[7/84] ADOBE_2017_10K
[8/84] ADOBE_2022_10K
[9/84] AES_2022_10K
[10/84] AMAZON_2017_10K
[11/84] AMAZON_2019_10K
[12/84] AMCOR_2020_10K
[13/84] AMCOR_2022_8K_dated-2022-07-01
[14/84] AMCOR_2023Q2_10Q
[15/84] AMCOR_2023Q4_EARNINGS
[16/84] AMCOR_2023_10K
[17/84] AMD_2015_10K
[18/84] AMD_2022_10K
[19/84] AMERICANEXPRESS_2022_10K
[20/84] AMERICANWATERWORKS_2020_10K
[21/84] AMERICANWATERWORKS_2021_10K
[22/84] AMERICANWATERWORKS_2022_10K
[23/84] BESTBUY_2017_10K
[24/84] BESTBUY_2019_10K
[25/8

In [2]:
import sys
sys.path.append("/content/financebench-rag/src")

## 1. Reader settings

Our reader model was chosen as Qwen3.5-4B in notebook 02. The following settings come from the same notebook:
- Thinking mode is off and decoding is greedy (rerun on the model gives the same answers)
- The prompt we chose asks the model to work step by step and end with `Final answer: <value>`
- Page images are rendered at 150 dpi

The following settings were newly added:
- If the answer is not on the pages, the model ends with `Final answer: not found` so that refusals can be counted separately from wrong answers
- Max token is increased from 512 (notebook 02) to 1024
- Each paeg is labelled with the name of its report, so the model can tell reports apart

Our prompt can be found in `src/financebench_rag/reader.py`.

In [4]:
import json
import time

import pandas as pd
import torch

from financebench_rag.data import REPO, gold_pages, load_questions, load_split
from financebench_rag.grading import numeric_match
from financebench_rag.reader import Reader

questions = load_questions()
split = load_split()
dev = [q for q in questions if q["financebench_id"] in set(split["dev"])]
retrieved = {fid: [tuple(page) for page in ranked[:5]] for fid, ranked in json.load(open(REPO / "results" / "dev_retrieval_hybrid.json")).items()}
print(f"{len(dev)} dev questions")

reader = Reader()
print(f"GPU memory after loading: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

30 dev questions


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

... (3 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

GPU memory after loading: 9.1 GB


## 2. Check that five page images fit

The cell below runs `retrieved_image` on 3 dev questions prints the time and the peak GPU memory used per answer, since 5 page images in one prompt is the largest input in the project.

If it runs out of memory, we might need to change the reader above to `Reader(dpi=100)` and run again.

In [5]:
torch.cuda.reset_peak_memory_stats()
for q in dev[:3]:
    result = reader.ask(q["question"], retrieved[q["financebench_id"]], "image")
    print(f"{q['financebench_id']}: {result['seconds']} s, {result['tokens']} tokens, final answer: {result['final']}")
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


financebench_id_04672: 28.3 s, 157 tokens, final answer: not found
financebench_id_01226: 53.5 s, 556 tokens, final answer: The operating margin change as of FY2022 for 3M was driven by total organic growth/productivity and other (1.0%), raw material impact (-2.4%), divestitures (-0.05%), and foreign exchange impacts (-0.39%). Operating margin is a useful metric for a company like 3M as it provides insight into the company's profitability and efficiency in generating profit from its core operations.
financebench_id_01865: 91.1 s, 1024 tokens, final answer: None
peak GPU memory: 12.4 GB


Peak GPU memory is 12.4 GB which fits within the 15 GB of the T4, so we can keep the images at 150 dpi.

## 3. Run all five settings

Each setting is saved as one file with every dev answer (since files on colab disappear when session ends).

A file holds the full response, the final answer, the number of generated tokens and the time of every dev question.

In [6]:
from google.colab import files

SETTINGS = ["gold_text", "gold_image", "retrieved_text", "retrieved_image", "closed_book"]


def pages_for(setting, q):
    if setting.startswith("gold"):
        return sorted(gold_pages(q))
    if setting.startswith("retrieved"):
        return retrieved[q["financebench_id"]]
    return []


answers = {}
for setting in SETTINGS:
    mode = "image" if setting.endswith("image") else "text"
    start = time.time()
    answers[setting] = {q["financebench_id"]: reader.ask(q["question"], pages_for(setting, q), mode) for q in dev}
    path = REPO / "results" / f"dev_answers_{setting}.json"
    path.write_text(json.dumps(answers[setting], indent=1) + "\n")
    files.download(str(path))
    print(f"{setting} done in {(time.time() - start) / 60:.1f} min")

print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

gold_text done in 21.6 min


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

gold_image done in 19.7 min


... (5 outputs left out)

retrieved_image done in 23.0 min


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

closed_book done in 25.0 min
peak GPU memory: 13.2 GB


## 4. Check the answers

The table shows, for each setting:
- the answers with no final answer line
- the answers cut off at 1024 tokens
- the `not found` answers
- the average answer length and time


In [ ]:
rows = []
for setting, result in answers.items():
    rs = list(result.values())
    mean_seconds = sum(r["seconds"] for r in rs) / len(rs)
    rows.append({
        "setting": setting,
        "no final answer line": sum(r["final"] is None for r in rs),
        "hit token limit": sum(r["tokens"] == reader.max_new_tokens for r in rs),
        "not found": sum(r["final"] is not None and "not found" in r["final"].lower() for r in rs),
        "mean tokens": round(sum(r["tokens"] for r in rs) / len(rs)),
        "mean seconds": round(mean_seconds, 1),
        "test hours (estimate)": round(mean_seconds * 120 / 3600, 1),
    })
pd.DataFrame(rows)

,setting,no final answer line,hit token limit,not found,mean tokens,mean seconds,test hours (estimate)
0,gold_text,2,3,3,575,43.1,1.4
1,gold_image,2,2,4,489,39.2,1.3
2,retrieved_text,7,7,9,650,53.0,1.8
3,retrieved_image,1,1,9,407,44.9,1.5
4,closed_book,7,7,2,676,50.0,1.7


In every setting, some answers were cut off at 1024 tokens (most in `retrieved_text` and `closed_book`).

The retrieved settings have the most `not found` answers.

## 5. Looking at the metrics questions

The 10 dev metrics questions have numeric answers, so they can be graded by code. According to the rule of Kobeissi and Langlais (2026), an answer is counted as correct when a number in its final answer is within 3% of a number in the reference answer.

In [10]:
metrics = [q for q in dev if q["question_type"] == "metrics-generated"]
display(pd.DataFrame(
    {"reference": [q["answer"] for q in metrics], **{s: [answers[s][q["financebench_id"]]["final"] for q in metrics] for s in SETTINGS}},
    index=[q["doc_name"] for q in metrics],
))

pd.Series(
    {s: sum(numeric_match(answers[s][q["financebench_id"]]["final"] or "", q["answer"]) for q in metrics) for s in SETTINGS},
    name="correct of 10",
)

,reference,gold_text,gold_image,retrieved_text,retrieved_image,closed_book
3M_2018_10K,$8.70,8.738,8.738,not found,not found,6.6
ACTIVISIONBLIZZARD_2019_10K,24.26,24.26,24.26,not found,not found,7.12
ADOBE_2017_10K,0.83,0.83,0.83,not found,not found,0.51
AMAZON_2017_10K,93.86,93.86,93.86,not found,not found,55.55
AMD_2015_10K,4.2%,4.18%,4.18%,4.18%,4.18%,not found
CORNING_2021_10K,10.3%,10.3,10.3,10.3,10.3,11.8
GENERALMILLS_2022_10K,0.54,0.54,0.54,None,0.56,0.36
MGMRESORTS_2020_10K,7.9%,7.9,7.9,None,10.0,None
PEPSICO_2021_10K,$4.60,4.625,4.625,4.625,4.625,4.3
WALMART_2020_10K,6.2%,6.2,6.2,None,6.2,None


,correct of 10
gold_text,10
gold_image,10
retrieved_text,3
retrieved_image,5
closed_book,0


- With the evidence pages, all 10 answers are correct in both text and image. 
- With retrieved pages, 3 text and 5 image answers are correct, and most missed are `not_found` ones.
- `closed_book` answers none corectly, so the model does not answer from memory.